# CoStar and Vexcel exploration notebook

Run the cells from top to bottom using the project Python environment. This copy preserves the saved analysis code and notes from `Costar analysis.ipynb`, with clearer sections and a lease dictionary preview. The original notebook is unchanged. Prior outputs are cleared so displayed results reflect your next run.

The CoStar data are a supplied snapshot. Vexcel's attribute catalog describes documented capabilities; API authentication and product access have not been verified here. Keys are read from `.env` and their values are never displayed.


## 1. Load API keys

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Find the project .env whether the kernel starts in the repo or notebook folder.
env_path = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No .env found in the working directory or its parents.")

load_dotenv(env_path, override=True)

vexcel_key = os.getenv("VEXCEL_API_KEY")
matterport_key = os.getenv("MATTERPORT_API_KEY")

print("Loaded environment from:", env_path)
print("Vexcel key found:", bool(vexcel_key))
print("Matterport key found:", bool(matterport_key))


In [ ]:
import json
import pandas as pd
from IPython.display import display


## Building lease data
The supplied CoStar snapshot includes recorded lease deals in `location-occupancy.jsonl`.
These are separate from asking rents in `commercial-listing-summary.jsonl`.
Not every building or tenancy has a lease deal, and individual terms may be missing.
Dates and open occupancies reflect the supplied snapshot, not a live availability check.
Apartment-building occupancies describe commercial tenants, not individual residents.


In [ ]:
# Locate the kit from either the project root or the notebook directory.
kit_dir = next(
    candidate
    for folder in (Path.cwd(), *Path.cwd().parents)
    for candidate in (folder, folder / "costar-hackathon-kit")
    if (candidate / "data" / "location-occupancy.jsonl").is_file()
)

def read_records(name):
    with (kit_dir / "data" / f"{name}.jsonl").open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

buildings = pd.read_csv(kit_dir / "commercial-properties.csv")
companies = {row["id"]: row.get("name") for row in read_records("company-detail")}
lease_rows = []
for occupancy in read_records("location-occupancy"):
    deal = occupancy.get("leaseDeal")
    if not deal:
        continue
    areas = [space["squareFeetOccupied"] for space in occupancy.get("spaces", [])
             if space.get("squareFeetOccupied") is not None]
    lease_rows.append({
        "occupancyId": occupancy["id"],
        "propertyId": occupancy["propertyId"],
        "tenantName": companies.get(occupancy.get("tenantLocationId")),
        "occupancyFrom": occupancy.get("fromDate"),
        "occupancyTo": occupancy.get("toDate"),
        "occupiedSquareFeet": sum(areas) if areas else None,
        "leaseDeal": deal,
    })
leases = pd.json_normalize(lease_rows)

def building_leases(property_id):
    """Return all recorded lease deals for a PropertyId, including historical deals."""
    return leases.loc[leases["propertyId"].eq(int(property_id))].copy()

# Search an address, then pass its PropertyId to building_leases(...).
def find_buildings(address_text):
    matches = buildings["Address"].str.contains(address_text, case=False, na=False, regex=False)
    return buildings.loc[matches, ["PropertyId", "Address", "City", "State"]].drop_duplicates()

print(f"Loaded {len(leases):,} lease deals across {leases['propertyId'].nunique():,} properties.")
# Example building with recorded lease data; replace with your chosen PropertyId.
example_property_id = int(leases["propertyId"].iloc[0])
print("Example PropertyId:", example_property_id)
display(building_leases(example_property_id)[[
    "propertyId", "tenantName", "occupiedSquareFeet",
    "leaseDeal.signDate", "leaseDeal.commencementDate", "leaseDeal.expirationDate",
    "leaseDeal.leaseTerm", "leaseDeal.leaseTermType",
    "leaseDeal.rateActual", "leaseDeal.rateBasis", "leaseDeal.rateAmountType",
    "leaseDeal.serviceType", "leaseDeal.monetaryUnit",
]].head(10))


Usage: `find_buildings("Main")`, then `building_leases(PROPERTY_ID)`.
All supplied lease-deal fields remain accessible under `leaseDeal.*`, including
`effectiveRate`, `effectiveRateBasis`, `freeMonths`, and `tenantImprovementAllowancePerSquareFeet`.
Do not compare raw rents without checking rate basis, amount type, currency, and service type;
zero/default values can mean unreported terms. See `models/location-occupancy.md` for field and enum definitions.


## 2. Lease dictionary and DataFrame previews

Each lease row is a tenancy with a recorded deal. A building can have multiple tenants and historical deals. `leases` retains all available nested deal fields; the preview below only selects the first five rows.


In [ ]:
display(leases.head())
dictionary_path = kit_dir / "lease-data-dictionary.csv"
if dictionary_path.is_file():
    lease_dictionary = pd.read_csv(dictionary_path)
    display(lease_dictionary)
else:
    print("Lease dictionary CSV is not present; consult models/location-occupancy.md.")


## 3. Load building records and count properties by market

In [ ]:
data_path = next(
    folder / "data" / "commercial-property.jsonl"
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / "costar-hackathon-kit")
    if (folder / "data" / "commercial-property.jsonl").is_file()
)

# Read JSON Lines, then flatten nested fields into columns.
records = pd.read_json(data_path, lines=True)
properties = pd.json_normalize(records.to_dict("records"))

market_names = {14: "Houston", 20: "Los Angeles"}

property_counts = (
    properties
    .groupby("primaryAddress.researchMarketId")["id"]
    .nunique()
    .rename(index=market_names)
    .rename_axis("market")
    .reset_index(name="property_count")
)

display(property_counts)
print("Total unique properties:", properties["id"].nunique())

## Date coverage of the supplied snapshot
This section runs independently and checks all occupancy records (including those without a lease deal), recorded lease dates, commercial listing dates, and sale dates. Each row summarizes one date field, not a complete or continuous history. Missing dates are counted separately from values that could not be parsed.

Lease expiration and occupancy end dates can be scheduled or use placeholder values. Their maximum does not identify the export date. Future sign, start, or sale dates also deserve review. No date filter is applied to the source records.


### Summarize date coverage

In [ ]:
coverage_kit_dir = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "costar-hackathon-kit")
    if (candidate / "data" / "location-occupancy.jsonl").is_file()
)
coverage_fields = {
    "location-occupancy": {
        "fromDate": "Occupancy period starts",
        "toDate": "Occupancy period ends (may be scheduled)",
        "moveInDate": "Tenant moves in",
        "leaseDeal.signDate": "Lease signed",
        "leaseDeal.commencementDate": "Lease term starts",
        "leaseDeal.expirationDate": "Scheduled lease expiration",
    },
    "commercial-listing-summary": {
        "createdDate": "Marketing listing created",
        "listingStatusUpdatedDate": "Listing status updated",
        "certifiedFreshDate": "Listing certified fresh",
    },
    "commercial-sale-comp": {"soldDate": "Property sold"},
}

def coverage_value(record, dotted_field):
    value = record
    for key in dotted_field.split("."):
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value

# Compare dates in UTC so mixed timezone offsets can be parsed consistently.
coverage_today = pd.Timestamp.now(tz="America/New_York").normalize().tz_convert("UTC")
coverage_rows = []
coverage_year_rows = []
for dataset, fields in coverage_fields.items():
    collected = {field: [] for field in fields}
    with (coverage_kit_dir / "data" / f"{dataset}.jsonl").open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            # Lease date denominators include only occupancies with a lease deal.
            for field in fields:
                if field.startswith("leaseDeal.") and not record.get("leaseDeal"):
                    continue
                collected[field].append(coverage_value(record, field))
    for field, meaning in fields.items():
        raw = pd.Series(collected[field], dtype="object")
        provided = raw.notna() & raw.astype(str).str.strip().ne("")
        dates = pd.to_datetime(raw.where(provided), format="mixed", errors="coerce", utc=True)
        valid = dates.dropna()
        earliest, latest = valid.min(), valid.max()
        coverage_rows.append({
            "dataset": dataset,
            "date_field": field,
            "meaning": meaning,
            "records_checked": len(raw),
            "valid_dates": len(valid),
            "missing_dates": int((~provided).sum()),
            "unparseable_dates": int((provided & dates.isna()).sum()),
            "earliest_UTC": earliest,
            "latest_UTC": latest,
            "span_years_approx": round((latest - earliest).total_seconds() / (365.25 * 86400), 2)
                if len(valid) else None,
            "dates_after_today": int((dates >= coverage_today + pd.Timedelta(days=1)).sum()),
        })
        for year, count in valid.dt.year.value_counts().sort_index().items():
            coverage_year_rows.append({"dataset": dataset, "date_field": field,
                                       "year_UTC": int(year), "record_count": int(count)})

date_coverage = pd.DataFrame(coverage_rows)
date_counts_by_year = pd.DataFrame(coverage_year_rows)
display(date_coverage)

# Year-by-year lease signing counts help show whether records cluster in certain years.
display(date_counts_by_year.loc[
    date_counts_by_year["date_field"].eq("leaseDeal.signDate")
].reset_index(drop=True))


Seems like there are historical lease deals

## Comprehensive variable catalogs: CoStar and Vexcel
**CoStar:** all eight supplied JSON schemas are traversed, including nested fields, array elements (`[]`), types, definitions, and numeric codes. This covers possible fields; inclusion does not guarantee a value exists in the actual records. Objects and their children have separate entries, so catalog entry counts exceed flat column counts. Nullable is explicit schema nullability, not a measurement of missing data.

**Vexcel:** a documented product-attribute overview follows. It is not a complete API schema or a verified inventory of your hackathon permissions. Imagery, coverage/collection metadata, and DSM/DTM elevation are also available through Vexcel services; those response fields depend on the endpoint. No API requests are made here.


### CoStar variable catalog

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

catalog_kit = next(
    candidate for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "costar-hackathon-kit")
    if (candidate / "schemas" / "commercial-property.schema.json").is_file()
)

# Read definitions, not just a sample record, so rarely populated fields are included.
def schema_catalog(schema, dataset):
    rows = []
    def resolve(node):
        if "$ref" in node:
            target = schema
            for piece in node["$ref"].removeprefix("#/").split("/"):
                target = target[piece.replace("~1", "/").replace("~0", "~")]
            return {**target, **{k:v for k,v in node.items() if k != "$ref"}}
        return node
    def visit(raw, path, ancestors=()):
        ref = raw.get("$ref")
        node = resolve(raw)
        options = [resolve(option) for option in node.get("anyOf", [])]
        non_null = [option for option in options if option.get("type") != "null"]
        if non_null:
            node = {**non_null[0], **{k:v for k,v in node.items() if k != "anyOf"}}
        kind = node.get("type", "unspecified")
        types = kind if isinstance(kind, list) else [kind]
        codes = "; ".join(f"{value} = {label}" for value, label in
                          zip(node.get("enum", []), node.get("x-enum-varnames", [])))
        if path:
            rows.append({"dataset": dataset, "variable": path,
                         "type": " / ".join(types),
                         "nullable": "null" in types or any(o.get("type") == "null" for o in options),
                         "meaning": node.get("description", ""), "code_values": codes})
        if ref and ref in ancestors:
            return
        next_ancestors = ancestors + ((ref,) if ref else ())
        for name, child in node.get("properties", {}).items():
            visit(child, f"{path}.{name}" if path else name, next_ancestors)
        if "items" in node:
            visit(node["items"], path + "[]", next_ancestors)
    visit(schema, "")
    return rows

catalog_rows = []
for schema_path in sorted((catalog_kit / "schemas").glob("*.schema.json")):
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    catalog_rows.extend(schema_catalog(schema, schema_path.name.removesuffix(".schema.json")))
costar_variables = pd.DataFrame(catalog_rows)
costar_overview = costar_variables.groupby("dataset").agg(
    catalog_entries=("variable", "size"),
    top_level_fields=("variable", lambda values: sum("." not in x and "[]" not in x for x in values)),
).reset_index()
display(costar_overview)

# All rows are displayed within a temporary setting; long definitions are wrapped by pandas.
def show_costar_variables(dataset=None, search=None):
    result = costar_variables
    if dataset:
        result = result.loc[result["dataset"].eq(dataset)]
    if search:
        match = (result["variable"].str.contains(search, case=False, regex=False, na=False)
                 | result["meaning"].str.contains(search, case=False, regex=False, na=False))
        result = result.loc[match]
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 160):
        display(result.reset_index(drop=True))
    return result.reset_index(drop=True)

# Start with every building field, including nested structures and array elements.
building_variable_catalog = show_costar_variables("commercial-property")


### Vexcel documented attribute catalog

In [ ]:
# Vexcel product concepts below are documentation labels, NOT exact API JSON keys.
# Availability depends on product permissions, location, collection, and API version.
vexcel_source = "https://vexceldata.com/intelligence/elements/property-attributes/"
vexcel_groups = {
    "Building geometry": ["Footprint polygon", "Footprint area", "Ground elevation", "Entrance elevation", "Primary structure", "Structure distance", "Volume below eave", "Number of stories"],
    "Roof": ["Condition", "Material", "Shape", "Centroid", "Elevation", "Maximum height", "Eave height", "Habitable attic"],
    "Roof objects": ["Solar detection", "Photovoltaic panels: count and area", "Thermal panels: count and area", "Air conditioning units", "Chimneys", "Satellite dishes", "Vents", "Skylights", "Tree cover"],
    "Roof condition details": ["Algae staining", "Vent staining", "Water pooling", "Dry water pooling", "Vegetation", "Organic matter", "Tarp detection and area", "Debris", "Rust", "Structural damage"],
    "Property and surroundings": ["Parcel geometry", "Parcel tree cover", "Building buffers: 5/30/100/200 ft", "Tree buffers: 5/30/100/200 ft", "Pool detection/types/counts", "Water slide", "Diving board", "Hardscapes", "Deck", "Enclosure", "Sport court", "Playground", "Trampoline", "Vehicle detection/types/counts"],
}
vexcel_variables = pd.DataFrame([
    {"category": category, "documented_attribute": attribute,
     "access_status": "Not verified for your key", "source": vexcel_source}
    for category, attributes in vexcel_groups.items() for attribute in attributes
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(vexcel_variables)

# Use on an actual JSON response to see its real field paths, including list elements.
# Does not make API calls or display response values / credentials.
def vexcel_response_fields(payload):
    paths = set()
    def walk(value, path=""):
        if isinstance(value, dict):
            for key, child in value.items():
                walk(child, f"{path}.{key}" if path else str(key))
        elif isinstance(value, list):
            paths.add((path, "list"))
            for child in value:
                walk(child, path + "[]")
        else:
            paths.add((path, type(value).__name__))
    walk(payload)
    return pd.DataFrame(sorted(paths), columns=["variable", "observed_type"])


### Browse any CoStar dataset
```python
show_costar_variables()  # All datasets (a large table)
show_costar_variables("location-occupancy")
show_costar_variables(search="rent")
show_costar_variables("commercial-property", search="parking")
```
`costar_variables` and `vexcel_variables` are ordinary DataFrames available in VS Code's Data Viewer. Definitions come directly from the supplied schemas and can be technical.

Vexcel sources: [Property attributes](https://vexceldata.com/intelligence/elements/property-attributes/) and [API overview](https://vexcel.atlassian.net/wiki/spaces/SUPPORT/pages/2136539158/Vexcel+API+-+Documentation). Exact accessible API variables remain unverified. If an API response is loaded into `response_json`, inspect its actual field names with `display(vexcel_response_fields(response_json))`.


### Search catalog examples

In [ ]:
show_costar_variables("commercial-property")
show_costar_variables("location-occupancy")
show_costar_variables(search="rent")

## DataFrames available in Data Viewer

- `properties`: building records, one row per property.
- `leases`: tenancy records with lease deals.
- `lease_dictionary`: variable definitions, when the CSV is present.
- `date_coverage`: minimum/maximum dates and missing-value counts.
- `date_counts_by_year`: date-field counts by year.
- `costar_variables`: schema definitions across eight CoStar datasets.
- `vexcel_variables`: documented Vexcel concepts, not actual API responses.

A repeated property ID in `leases` is expected. Date ranges do not establish complete historical coverage. Normalize rent units and expense terms before comparing deals.


# Lease validation: trace the dashboard back to CoStar
Run this section from its first code cell; it is independent of earlier notebook variables. No API key, live server, downloads, or database rebuild is required.

**Question:** Are the source joins, inferred lease outcomes, and reported renewal rates reproducible? This section checks the lease side. It does not establish that parking causes or predicts renewal.

A raw row is a tenancy; `leaseDeal` may be missing. The processed CSV's `lease_id` is actually the raw **occupancy ID**, not `leaseDeal.id`. Outcomes are inferred by the preparation script, not verified signed renewal contracts. The app counts renewed + expanded as success and downsized + moved_out as failure.

In [ ]:
from pathlib import Path
import json
from collections import defaultdict
import pandas as pd
from IPython.display import display, Markdown, HTML

validation_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "backend/parallax/events.py").is_file())
validation_data = validation_root / "data/real"
validation_kit = validation_root / "costar-hackathon-kit"
def validation_read(name):
    with (validation_kit / "data" / f"{name}.jsonl").open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
validation_raw = validation_read("location-occupancy")
validation_properties = pd.json_normalize(validation_read("commercial-property"))
validation_companies = pd.DataFrame([
    {"tenantLocationId": r["id"], "sourceTenantName": r.get("name")}
    for r in validation_read("company-detail")
])
# Generate missing processed tables using the same preparation script as the app.
# No Vexcel/API calls are made by this step.
import os
import sys
import shutil
import subprocess

validation_required = ["costar_properties.csv", "costar_leases.csv",
                       "costar_buildings.csv", "costar_occupancy.csv"]
validation_missing = [name for name in validation_required
                      if not (validation_data / name).is_file()]
if validation_missing:
    print("Missing processed CoStar files:", validation_missing)
    print("Generating them from the supplied raw JSONL files...")
    validation_data.mkdir(parents=True, exist_ok=True)
    for name in ["commercial-property", "company-detail", "location-occupancy"]:
        destination = validation_data / f"{name}.jsonl"
        source = validation_kit / "data" / f"{name}.jsonl"
        if not destination.is_file():
            if not source.is_file():
                raise FileNotFoundError(f"Raw CoStar file is missing: {source}")
            shutil.copy2(source, destination)
    preparation_env = os.environ.copy()
    preparation_env["PARALLAX_DATA_MODE"] = "real"
    preparation_env["PARALLAX_DATA_DIR"] = str(validation_root / "data")
    preparation = subprocess.run(
        [sys.executable, "-m", "scripts.prepare_costar"],
        cwd=validation_root / "backend", env=preparation_env,
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    print(preparation.stdout)
    if preparation.returncode:
        raise RuntimeError("CoStar preparation failed:\n" + preparation.stderr)
    still_missing = [name for name in validation_required
                     if not (validation_data / name).is_file()]
    if still_missing:
        raise FileNotFoundError(f"Preparation did not create {still_missing} in {validation_data}")
print("Processed CoStar source:", validation_data.resolve())

validation_processed = pd.read_csv(validation_data / "costar_leases.csv")
validation_prep_properties = pd.read_csv(validation_data / "costar_properties.csv")
validation_imaging = pd.read_csv(validation_data / "costar_buildings.csv")
# Read-only snapshot of exactly the tables used by the saved real-mode app database.
import duckdb
with duckdb.connect(str(validation_root / "data/parallax_real.duckdb"), read_only=True) as con:
    validation_model_leases = con.execute("SELECT * FROM leases").df()
    validation_model_buildings = con.execute("SELECT * FROM buildings").df()
    validation_usage = con.execute("SELECT * FROM usage").df()
validation_app_paths = sorted((validation_root / "data/results").glob("parallax-app-results-*.json"))
validation_app = json.loads(validation_app_paths[-1].read_text(encoding="utf-8")) if validation_app_paths else None
validation_saved = json.loads((validation_data / "run_summary.json").read_text(encoding="utf-8")) if (validation_data / "run_summary.json").is_file() else None
pd.set_option("display.max_columns", 18)
print("Sources:", validation_data)
print("Saved app report:", validation_app_paths[-1].name if validation_app_paths else "not present")

## 1. Understand the populations and identifiers
These are different populations, not interchangeable denominators. Buildings without a lease deal are still real buildings. Preparation drops properties without a usable location resolved through tenant-company addresses. Imaging selects a further subset. The app loader then keeps lease events on imaged CoStar buildings.

In [ ]:
validation_inventory = pd.DataFrame([
    ["Raw buildings", len(validation_properties), validation_properties["id"].nunique(), "One building/property"],
    ["Raw tenancies", len(validation_raw), len({r["propertyId"] for r in validation_raw}), "One occupancy record, with or without a lease deal"],
    ["Tenancies with lease deals", sum(bool(r.get("leaseDeal")) for r in validation_raw), len({r["propertyId"] for r in validation_raw if r.get("leaseDeal")}), "One occupancy with deal terms"],
    ["Prepared located properties", len(validation_prep_properties), validation_prep_properties["property_id"].nunique(), "Property resolved through the preparation join"],
    ["Prepared expiry events", len(validation_processed), validation_processed["property_id"].nunique(), "Expiry with an inferred outcome"],
    ["Properties selected for imaging", len(validation_imaging), validation_imaging["building_id"].nunique(), "Selected non-apartment imaging candidate"],
    ["App database lease events", len(validation_model_leases), validation_model_leases["building_id"].nunique(), "Prepared event at an imaged building"],
], columns=["population", "rows", "unique_properties", "row_meaning"])
display(validation_inventory)

## 2. Inspect the source records and verify joins
Join buildings using `propertyId` to `commercial-property.id`; join tenant companies using `tenantLocationId` to `company-detail.id`. Below, source names and dates are retained beside the processed outcomes. Missing tenant IDs mean unknown tenants.

The checks establish join integrity and agreement with source records, not that source facts are independently correct. Sample records are selected across outcome groups rather than only showing successful renewals.

In [ ]:
def validation_date(value):
    return pd.Timestamp(str(value)[:10]) if value else pd.NaT
validation_source_rows = []
for r in validation_raw:
    deal = r.get("leaseDeal") or {}
    validation_source_rows.append({
        "occupancyId": r["id"], "propertyId": r["propertyId"],
        "tenantLocationId": r.get("tenantLocationId"),
        "sourceDealId": deal.get("id"), "sourceReason": r.get("expirationReason"),
        "sourceFrom": validation_date(r.get("fromDate")),
        "sourceTo": validation_date(r.get("toDate")),
        "sourceSign": validation_date(deal.get("signDate")),
        "sourceCommencement": validation_date(deal.get("commencementDate")),
        "sourceExpiration": validation_date(deal.get("expirationDate")),
        "sourceSqft": sum((s.get("squareFeetOccupied") or 0) for s in (r.get("spaces") or [])),
    })
validation_sources = pd.DataFrame(validation_source_rows)
validation_audit = validation_processed.merge(validation_sources, left_on="lease_id", right_on="occupancyId", how="left", validate="many_to_one", indicator="source_join")
validation_audit = validation_audit.merge(validation_companies, on="tenantLocationId", how="left", validate="many_to_one")
validation_audit["csv_expiry"] = pd.to_datetime(validation_audit["expiry_date"], errors="coerce")
validation_join_checks = pd.DataFrame([
    ["Duplicate raw occupancy IDs", int(validation_sources["occupancyId"].duplicated().sum())],
    ["Duplicate processed occupancy IDs", int(validation_processed["lease_id"].duplicated().sum())],
    ["Processed events missing raw tenancy", int(validation_audit["source_join"].ne("both").sum())],
    ["Property ID differs from source", int(validation_audit["property_id"].ne(validation_audit["propertyId"]).sum())],
    ["Source property absent from building file", int((~validation_audit["propertyId"].isin(validation_properties["id"])).sum())],
    ["Expiration differs from source", int(validation_audit["csv_expiry"].ne(validation_audit["sourceExpiration"]).sum())],
    ["Known tenant ID without company match", int((validation_audit["tenantLocationId"].notna() & ~validation_audit["tenantLocationId"].isin(validation_companies["tenantLocationId"])).sum())],
    ["Unknown tenant ID (review, not automatically an error)", int(validation_audit["tenantLocationId"].isna().sum())],
], columns=["check", "affected_rows"])
display(validation_join_checks)
display(validation_audit.groupby("outcome", group_keys=False).head(2)[[
    "occupancyId", "sourceDealId", "propertyId", "sourceTenantName", "sourceSign",
    "sourceCommencement", "sourceExpiration", "sourceTo", "sourceReason", "outcome"
]])

## 3. Reproduce the outcome-label rules
The pipeline fixes its reference date at **November 1, 2026** and requires expiration at least 90 days earlier. Its supported expiry window starts January 1, 2016. This is an implementation cutoff, not today's date.

Rules below mirror `backend/scripts/prepare_costar.py`, including the first matching follow-on tenancy in file order. "Renewed" can be inferred simply because an occupancy is still open. Follow-on searches use dates within 180 or 365 days before or after of the occupancy end, not necessarily a later start. Unknown tenant IDs can match other unknown tenants in the same building. These are assumptions to audit, not proof of a signed renewal.

This reproduction checks the rules are applied consistently; it is not an independent ground-truth validation.

In [ ]:
validation_pairs = defaultdict(list)
for r in validation_raw:
    validation_pairs[(r["propertyId"], r.get("tenantLocationId"))].append(r)
validation_located = set(validation_prep_properties["property_id"])
validation_moved = {1, 3, 4, 5, 6, 10, 13}
validation_stayed = {7, 12}
validation_cutoff = pd.Timestamp("2026-11-01") - pd.Timedelta(days=90)
def validation_size(r):
    return sum(s.get("squareFeetOccupied") or 0 for s in (r.get("spaces") or []))
def validation_classify(r):
    deal = r.get("leaseDeal") or {}
    expiry, end = validation_date(deal.get("expirationDate")), validation_date(r.get("toDate"))
    base = {"occupancyId": r["id"], "reproduced_outcome": None, "rule": "", "followonId": None}
    def finish(rule, outcome=None, nxt=None):
        return {**base, "rule": rule, "reproduced_outcome": outcome, "followonId": nxt["id"] if nxt else None}
    if pd.isna(expiry): return finish("excluded: missing expiration")
    if not pd.Timestamp("2016-01-01") <= expiry <= validation_cutoff:
        return finish("excluded: outside preparation date window")
    if r["propertyId"] not in validation_located: return finish("excluded: property has no prepared location")
    reason, area = r.get("expirationReason"), validation_size(r)
    pair = validation_pairs[(r["propertyId"], r.get("tenantLocationId"))]
    def followons(days):
        return [x for x in pair if x is not r and x.get("fromDate")
                and abs((validation_date(x["fromDate"]) - end).days) <= days]
    def sized(nxt):
        after = validation_size(nxt) if nxt else area
        return "downsized" if area and after < .85*area else "expanded" if area and after > 1.15*area else "renewed"
    nxt = None
    if pd.notna(end) and reason in validation_moved:
        label, rule = "moved_out", "move-out reason code"
    elif pd.notna(end) and reason == 8:
        label, rule = "downsized", "shed-space reason code"
    elif pd.notna(end) and reason in validation_stayed:
        matches = followons(365); nxt = matches[0] if matches else None
        label, rule = sized(nxt), "renewal/same-building reason; area comparison when matched"
    elif pd.notna(end):
        matches = followons(180); nxt = matches[0] if matches else None
        if nxt: label, rule = sized(nxt), "follow-on occupancy within +/-180 days"
        elif end > expiry + pd.Timedelta(days=365): label, rule = "renewed", "occupancy ends over 365 days after expiry"
        elif end <= expiry + pd.Timedelta(days=180): label, rule = "moved_out", "occupancy ends by expiry +180 days"
        else: return finish("excluded: ambiguous end date")
    else:
        label, rule = "renewed", "open occupancy assumed renewed"
    if pd.notna(end) and reason in validation_moved and end < expiry-pd.Timedelta(days=365):
        return finish("excluded: moved out over a year before expiry")
    return finish(rule, label, nxt)
validation_labels = pd.DataFrame(validation_classify(r) for r in validation_raw)
validation_audit = validation_audit.merge(validation_labels, on="occupancyId", validate="one_to_one")
validation_audit["label_matches"] = validation_audit["outcome"].eq(validation_audit["reproduced_outcome"])
display(validation_labels.groupby("rule", dropna=False).size().rename("raw_tenancies").reset_index())
display(pd.crosstab(validation_audit["outcome"], validation_audit["reproduced_outcome"], margins=True))
print(f"Reproduced labels agree for {validation_audit['label_matches'].sum():,}/{len(validation_audit):,} prepared events.")
validation_label_examples = validation_audit.groupby("rule", group_keys=False).head(1)
display(validation_label_examples[["occupancyId", "propertyId", "sourceTenantName", "sourceExpiration", "sourceTo", "sourceReason", "followonId", "rule", "outcome", "label_matches"]])

## 4. Inspect date coverage and missing information
Dates are summarized for the prepared expiry events, not all historical CoStar records. Future contractual dates are not evidence of future data collection. Null occupancy end dates indicate open records in this snapshot. Missing signing dates do not prevent the app from using an expiration event.

In [ ]:
validation_date_summary = pd.DataFrame([
    {"field": col, "earliest": validation_audit[col].min(), "latest": validation_audit[col].max(),
     "reported": int(validation_audit[col].notna().sum()), "missing": int(validation_audit[col].isna().sum())}
    for col in ["sourceSign", "sourceCommencement", "sourceExpiration", "sourceFrom", "sourceTo"]
])
display(validation_date_summary)
validation_date_flags = pd.DataFrame([
    ["Sign after commencement (review, not automatically invalid)", int((validation_audit.sourceSign > validation_audit.sourceCommencement).sum())],
    ["Commencement after expiration", int((validation_audit.sourceCommencement > validation_audit.sourceExpiration).sum())],
    ["Unknown tenant used in follow-on match", int((validation_audit.tenantLocationId.isna() & validation_audit.followonId.notna()).sum())],
    ["Renewed by open-occupancy assumption", int(validation_audit.rule.eq("open occupancy assumed renewed").sum())],
    ["Nonpositive recorded occupied area", int(validation_audit.sourceSqft.le(0).sum())],
], columns=["review_flag", "rows"])
display(validation_date_flags)

## 5. Calculate renewal rates without using parking measurements
**App definition:** (renewed + expanded) / all eligible outcomes. **Broader retained-tenant definition:** also includes downsized. Neither is a verified contract-renewal rate.

The second measure below changes the success definition only. It does not repair uncertain outcome labels. Groups with small samples should not be compared confidently. Rates are event-weighted, not area-weighted or building-weighted.

In [ ]:
def validation_rates(frame, label):
    n = len(frame)
    kept = int(frame["outcome"].isin(["renewed", "expanded"]).sum())
    retained = int(frame["outcome"].isin(["renewed", "expanded", "downsized"]).sum())
    return {"population": label, "events": n, "renewed_or_expanded": kept,
            "app_rate_pct": round(100*kept/n,2) if n else None,
            "retained_including_downsized": retained,
            "broader_retention_pct": round(100*retained/n,2) if n else None}
validation_outcomes = validation_processed["outcome"].value_counts().rename_axis("outcome").reset_index(name="events")
validation_outcomes["percent"] = (100*validation_outcomes.events/len(validation_processed)).round(2)
display(validation_outcomes)
validation_sensitivity = pd.DataFrame([
    validation_rates(validation_audit, "All prepared expiry events"),
    validation_rates(validation_audit.loc[~validation_audit.rule.eq("open occupancy assumed renewed")], "Exclude open-occupancy assumptions"),
    validation_rates(validation_audit.loc[validation_audit.tenantLocationId.notna()], "Known tenant IDs only"),
    validation_rates(validation_audit.loc[validation_audit.sourceReason.isin(validation_moved | validation_stayed | {8}) & validation_audit.sourceTo.notna()], "Informative reason codes with closed occupancy only"),
])
display(validation_sensitivity)
validation_yearly_rates = pd.DataFrame([
    {"expiry_year": int(year), **validation_rates(g, "Prepared events")}
    for year,g in validation_audit.groupby(validation_audit.sourceExpiration.dt.year)
])
validation_city_rates = pd.DataFrame([
    validation_rates(g, city) for city,g in validation_audit.merge(
        validation_prep_properties[["property_id", "city"]], on="property_id", validate="many_to_one").groupby("city")
])
display(validation_yearly_rates)
display(validation_city_rates)
# Visual bars keep numerator/denominator visible in the table above.
from html import escape
validation_bars = ''.join(
    f'<tr><td>{escape(str(r.expiry_year))}</td><td><div style="background:#4878a8;width:{r.app_rate_pct*3:.1f}px;height:16px"></div></td><td>{r.app_rate_pct:.1f}% ({r.renewed_or_expanded}/{r.events})</td></tr>'
    for r in validation_yearly_rates.itertuples()
)
display(HTML('<h4>Prepared-event rate by expiration year</h4><table>'+validation_bars+'</table>'))
print("Sensitivity rows change both the population and its composition; differences are not causal effects.")

## 6. Reconcile the dashboard's smaller sample
This is the boundary with the parking analysis. We read existing usage values only to reproduce the app's eligibility filter; we do not validate car detection here. The app uses expiry dates from March 1, 2019 (inclusive) to October 1, 2026 (exclusive), then removes non-surface/unmeasurable properties and events without usable prior usage.

The original preparation cutoff is earlier than October 2026, so the effective date coverage can be narrower. Parking labels also include a heuristic that flags very low car counts, not independently confirmed garages.

In [ ]:
import numpy as np
validation_events = validation_model_leases.loc[
    validation_model_leases.expiry_date.ge("2019-03-01") & validation_model_leases.expiry_date.lt("2026-10-01")
].merge(validation_model_buildings[["building_id", "parking_type"]], on="building_id", validate="many_to_one")
validation_eligible_start = len(validation_events)
validation_garage_count = int(validation_events.parking_type.ne("surface").sum())
validation_events = validation_events.loc[validation_events.parking_type.eq("surface")].copy()
validation_series = {
    bid: g.sort_values("capture_date") for bid,g in validation_usage.loc[
        validation_usage.status.eq("ok") | validation_usage.index_smooth.notna()
    ].groupby("building_id")
}
validation_event_usage=[]
for r in validation_events.itertuples():
    g=validation_series.get(r.building_id)
    if g is None: validation_event_usage.append(np.nan); continue
    window=g.loc[g.capture_date.ge(r.expiry_date-pd.DateOffset(months=18)) & g.capture_date.lt(r.expiry_date), "index_smooth"].dropna()
    validation_event_usage.append(window.mean() if len(window) else np.nan)
validation_events["usage"] = validation_event_usage
validation_no_usage = int(validation_events.usage.isna().sum())
validation_dashboard_events = validation_events.loc[validation_events.usage.notna()].copy()
validation_dashboard_events["usage"] = validation_dashboard_events.usage.clip(20,200)
validation_funnel = pd.DataFrame([
    ["All prepared expiry events",len(validation_processed)],
    ["Lease events retained in app database",len(validation_model_leases)],
    ["App expiry window",validation_eligible_start],
    ["After parking-type exclusion",validation_eligible_start-validation_garage_count],
    ["After prior-usage exclusion",len(validation_dashboard_events)],
],columns=["stage","remaining_events"])
display(validation_funnel)
validation_expected_exclusions={"total_expiries":validation_eligible_start,"garage_not_measurable":validation_garage_count,"no_usable_imagery":validation_no_usage,"usable_events":len(validation_dashboard_events)}
if validation_saved:
    display(pd.DataFrame([{"check":key,"recalculated":value,"saved":validation_saved["exclusions"].get(key),"matches":value==validation_saved["exclusions"].get(key)} for key,value in validation_expected_exclusions.items()]))
display(pd.DataFrame([validation_rates(validation_processed,"All prepared events"),validation_rates(validation_dashboard_events,"Dashboard eligible events")]))

## 7. Show the arithmetic behind the app's renewal-by-usage table
The same eligible events are divided into usage quintiles. Each displayed rate is success count / event count. Wilson intervals describe uncertainty for a binomial rate; they do not account for repeated tenants/buildings or measurement/label errors. Matching these calculations demonstrates reproducibility, not predictive validity.

In [ ]:
validation_bin_rows=[]
validation_bin_ids=pd.qcut(validation_dashboard_events["usage"],5,labels=False,duplicates="drop")
for bin_id,g in validation_dashboard_events.groupby(validation_bin_ids):
    n=len(g);k=int(g.outcome.isin(["renewed","expanded"]).sum());p=k/n;z=1.96
    denominator=1+z*z/n
    center=(p+z*z/(2*n))/denominator
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/denominator
    validation_bin_rows.append({"bin":int(bin_id)+1,"events":n,"successes":k,"usage_min":g.usage.min(),"usage_max":g.usage.max(),"renewal_rate":k/n,"ci_low":center-half,"ci_high":center+half})
validation_bin_math=pd.DataFrame(validation_bin_rows)
display(validation_bin_math.round(3))
if validation_app:
    validation_app_bins=pd.DataFrame(validation_app["results"]["bins"].get("bins",[]))
    validation_bin_comparison=validation_bin_math.merge(validation_app_bins[["bin","n","renewal_rate","ci_low","ci_high"]],on="bin",suffixes=("_manual","_app"),validate="one_to_one")
    validation_bin_comparison["matches_app_rounding"]=(validation_bin_comparison.events.eq(validation_bin_comparison.n)
        & validation_bin_comparison.renewal_rate_manual.round(3).eq(validation_bin_comparison.renewal_rate_app)
        & validation_bin_comparison.ci_low_manual.round(3).eq(validation_bin_comparison.ci_low_app)
        & validation_bin_comparison.ci_high_manual.round(3).eq(validation_bin_comparison.ci_high_app))
    display(validation_bin_comparison[["bin","events","successes","renewal_rate_manual","renewal_rate_app","matches_app_rounding"]])

## 8. Validation results and limitations
This is an audit of records and arithmetic, not proof of the teammate's hypothesis. The regression and occupancy findings below are loaded from the saved app response; they are **not independently refitted in this section**. Parking detection, usage construction, model assumptions, train/test predictions, and the full validation suite require separate review with the imagery/model teammates.

The preparation script exports raw contract rent as `rent_psf` without carrying rate-basis information into that CSV. The loader fills missing rents with the sample median (or a fallback). Do not use this processed column for a rent time series without returning to original units and missingness; it also affects interpretation of the regression's rent control.

In [ ]:
validation_conclusions = pd.DataFrame([
    ["Processed-to-source join", f"{int(validation_audit.source_join.eq('both').sum()):,}/{len(validation_audit):,} matched", "Checks record linkage"],
    ["Outcome-rule reproduction", f"{int(validation_audit.label_matches.sum()):,}/{len(validation_audit):,} agree", "Agreement is not independent ground truth"],
    ["Open-record renewal assumptions", str(int(validation_audit.rule.eq('open occupancy assumed renewed').sum())), "Needs manual source review"],
    ["Dashboard eligible sample", str(len(validation_dashboard_events)), "Selection depends on available parking data"],
], columns=["check","result","interpretation"])
display(validation_conclusions)
if validation_app:
    validation_stat_rows=[]
    for name,key,estimate in [("Renewal regression","regression","odds_ratio_per_10pt_drop"),("Occupancy link","occ_link","usage_pts_per_10pt_occupancy")]:
        r=validation_app["results"][key]
        validation_stat_rows.append({"analysis":name,"estimate":r.get(estimate),"CI_low":r.get("or_ci_low",r.get("ci_low")),"CI_high":r.get("or_ci_high",r.get("ci_high")),"p_value":r.get("p_value_usage",r.get("p_value")),"sample_n":r.get("n")})
    display(pd.DataFrame(validation_stat_rows))
    display(Markdown("**Interpretation:** a renewal odds-ratio interval containing 1 does not establish a clear renewal association. An occupancy coefficient interval containing 0 does not establish a clear occupancy association. These checks do not prove there is no relationship."))
    print("Saved occupancy placebo result:",validation_app["results"]["occ_link"].get("placebo"))
print("Useful Data Viewer tables: validation_audit, validation_labels, validation_sensitivity, validation_funnel, validation_bin_math.")

# Extended validation: occupancy, regression, and prediction
Run the first setup cell in this section, then the following cells in order. These cells read the existing real database, refit the models, and save results locally. They do not request images, call an LLM, rebuild the database, or change the dashboard.

**What consistent classification means:** the earlier audit reproduced all 4,314 processed labels using the same rules. This establishes implementation agreement, not that every label describes a real signed renewal. The table below explains those rules. Some statistical checks use parking inputs; their measurement accuracy remains a separate computer-vision task.

In [ ]:
import os, sys, json, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display, Markdown, HTML

extended_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend/parallax/events.py").is_file())
os.environ["PARALLAX_DATA_MODE"] = "real"
os.environ["PARALLAX_DATA_DIR"] = str(extended_root / "data")
if str(extended_root / "backend") not in sys.path:
    sys.path.insert(0, str(extended_root / "backend"))
from parallax import config, analysis, occupancy, events
importlib.reload(config)
# Read the database directly; do not let get_store() silently generate synthetic data.
with duckdb.connect(str(extended_root / "data/parallax_real.duckdb"), read_only=True) as con:
    extended_tables = {name: con.execute(f"SELECT * FROM {name}").df()
                       for name in ["buildings", "leases", "usage", "tenancies", "captures"]}
ext_b, ext_l, ext_u, ext_t = (extended_tables[k] for k in ["buildings", "leases", "usage", "tenancies"])
ext_events, ext_exclusions = events.build_lease_events(ext_b, ext_l, ext_u, window_months=18)
ext_panel = occupancy.build_panel(ext_b, ext_u, ext_t)
ext_app_files = sorted((extended_root / "data/results").glob("parallax-app-results-*.json"))
ext_saved_app = json.loads(ext_app_files[-1].read_text(encoding="utf-8")) if ext_app_files else None
print(f"Real database: {len(ext_b):,} buildings; {len(ext_events):,} eligible expiry events; {len(ext_panel):,} occupancy/capture observations.")
ext_outputs = {}

## 9. Explain the renewal rules in plain language
Rules run in order after expiry/date and location eligibility checks. The reference date is November 1, 2026 and the expiry must be at least 90 days earlier. A move-out more than a year before expiry is excluded when a move-out reason applies.

For matched follow-on records, area below 85% of the old area means downsized; above 115% means expanded; otherwise renewed. The first match in source order is used. These are heuristics, not contract verification. The earlier section shows actual source records, exact rule counts, and agreement checks.

In [ ]:
renewal_rule_guide = pd.DataFrame([
    ["Closed occupancy and move-out reason", "moved_out", "Reason codes 1, 3, 4, 5, 6, 10, 13"],
    ["Closed occupancy and shed-space reason", "downsized", "Reason code 8"],
    ["Renewal/same-building reason", "Area comparison, otherwise renewed", "Codes 7 or 12; search +/-365 days around end"],
    ["Other closed occupancy with follow-on record", "Area comparison", "Same building and tenant; search +/-180 days around end"],
    ["No follow-on; occupancy ends >365 days after expiry", "renewed", "Assumes continued occupancy means renewal"],
    ["No follow-on; occupancy ends by expiry +180 days", "moved_out", "Assumes departure near/before expiry means nonrenewal"],
    ["Occupancy end missing", "renewed", "Open-record assumption; not a signed-contract observation"],
    ["Other cases", "Excluded", "Ambiguous outcome"],
], columns=["condition", "assigned_outcome", "interpretation"])
display(renewal_rule_guide)
display(pd.crosstab(ext_events.outcome, ext_events.renewed.rename("app_success"), margins=True))
print("Success=1: renewed or expanded. Downsized is success=0, even if the tenant stayed.")

## 10. Recalculate occupancy from tenancy dates and area
At each imagery date: active occupied area / app building area x 100. A tenancy is active when start <= date and end is missing or end >= date. The app caps the result at 120%, not 100%. Values above 100% are a data-quality signal, not literal physical occupancy.

The denominator can be the reported rentable area, maximum tenant area, or 50,000 square feet when missing. The table identifies which denominator was actually used. Overlapping tenancies can inflate the numerator; overlapping records for a known tenant are review flags, not automatic duplicates, because a tenant may occupy multiple legitimate spaces.

In [ ]:
ext_sizes = ext_b.set_index("building_id")["sqft"]
ext_t_groups = {bid:g for bid,g in ext_t.groupby("building_id")}
occupancy_checks=[]
for r in ext_panel.itertuples():
    g=ext_t_groups[r.building_id]
    active=g.loc[g.from_date.le(r.capture_date) & (g.to_date.isna() | g.to_date.ge(r.capture_date))]
    area=float(active.sqft.sum()); size=float(ext_sizes[r.building_id])
    raw_pct=100*area/size if size>0 else np.nan
    occupancy_checks.append({"building_id":r.building_id,"capture_date":r.capture_date,
        "active_tenancies":len(active),"occupied_sqft":area,"building_sqft":size,
        "uncapped_pct":raw_pct,"manual_capped_pct":min(raw_pct,120),"app_pct":r.occ_0})
occupancy_audit=pd.DataFrame(occupancy_checks)
occupancy_audit["matches_app"]=np.isclose(occupancy_audit.manual_capped_pct,occupancy_audit.app_pct,equal_nan=True)
display(occupancy_audit.head(15).round(3))
prep_sizes=pd.read_csv(extended_root / "data/real/costar_properties.csv")
denominator_audit=ext_b[["building_id","sqft"]].merge(prep_sizes[["building_id","rba_sqft","max_tenant_sqft"]],on="building_id",how="left",validate="one_to_one")
rba=pd.to_numeric(denominator_audit.rba_sqft,errors="coerce")
max_area=pd.to_numeric(denominator_audit.max_tenant_sqft,errors="coerce")
denominator_audit["denominator_source"]=np.select([rba.notna(),max_area.notna()],["reported rentable area","maximum tenant area"],default="fallback: 50,000 sqft")
display(denominator_audit.groupby("denominator_source").size().rename("buildings").reset_index())
occupancy_quality=pd.DataFrame([
    ["Manual occupancy agrees with app",int(occupancy_audit.matches_app.sum()),len(occupancy_audit)],
    ["Uncapped occupancy above 100%",int(occupancy_audit.uncapped_pct.gt(100).sum()),len(occupancy_audit)],
    ["Uncapped occupancy above 120%",int(occupancy_audit.uncapped_pct.gt(120).sum()),len(occupancy_audit)],
    ["App building sizes nonpositive or missing",int((ext_b.sqft.le(0)|ext_b.sqft.isna()).sum()),len(ext_b)],
],columns=["check","affected_or_matching","denominator"])
display(occupancy_quality)
# Review potential overlaps in raw prepared tenancies, retaining tenant IDs.
overlap_source=pd.read_csv(extended_root / "data/real/costar_occupancy.csv",parse_dates=["from_date","to_date"])
overlap_source=overlap_source.loc[overlap_source.building_id.isin(ext_b.building_id) & overlap_source.tenant_location_id.notna()].sort_values(["building_id","tenant_location_id","from_date"])
review_overlap=[]
for (bid,tid),g in overlap_source.groupby(["building_id","tenant_location_id"]):
    latest_end=pd.NaT; previous_id=None
    for row in g.itertuples():
        end=row.to_date if pd.notna(row.to_date) else pd.Timestamp("2200-01-01")
        if pd.notna(latest_end) and row.from_date<=latest_end:
            review_overlap.append({"building_id":bid,"tenant_location_id":tid,"occupancy_id":row.occupancy_id,"previous_occupancy_id":previous_id,"start":row.from_date,"end":row.to_date})
        if pd.isna(latest_end) or end>latest_end:latest_end=end;previous_id=row.occupancy_id
occupancy_overlap_review=pd.DataFrame(review_overlap)
print("Known-tenant records overlapping an earlier record in the same building:",len(occupancy_overlap_review))
display(occupancy_overlap_review.head(15))
ext_outputs["occupancy_quality"]=occupancy_quality.to_dict("records")

## 11. Refit the occupancy association and timing checks
The occupancy model compares changes within each building while controlling for capture year. Building-clustered standard errors account for repeated observations of the same building. It tests association, not causation.

The coefficient is the change in parking-index points associated with 10 percentage points of occupancy. Confidence intervals containing zero do not establish a clear association. Lead/lag comparisons and the sharp-drop comparison are exploratory and use different eligible observations. The placebo uses 30 seeded shuffles; one pass/fail result is not a complete validation.

In [ ]:
occupancy_refit=occupancy.link(ext_panel)
occupancy_timing=occupancy.lead_lag(ext_panel)
occupancy_drop=occupancy.drop_followthrough(ext_panel)
occupancy_placebo=occupancy.placebo(ext_panel,n_iter=30,seed=2)
display(pd.DataFrame([{k:v for k,v in occupancy_refit.items() if k not in ["bins","interpretation"]}]))
display(pd.DataFrame(occupancy_timing["offsets"]))
display(pd.DataFrame([occupancy_drop]))
display(pd.DataFrame([occupancy_placebo]))
print(occupancy_refit.get("interpretation",occupancy_refit.get("error")))
if ext_saved_app:
    old=ext_saved_app["results"]["occ_link"]
    display(pd.DataFrame([{"metric":k,"refit":occupancy_refit.get(k),"saved_app":old.get(k)} for k in ["n","n_buildings","usage_pts_per_10pt_occupancy","ci_low","ci_high","p_value"]]))
ext_outputs.update(occupancy_refit=occupancy_refit,occupancy_timing=occupancy_timing,occupancy_drop=occupancy_drop,occupancy_placebo=occupancy_placebo)

## 12. Independently refit the full lease-renewal regression
The response is the inferred success label (renewed or expanded). Predictors are usage, usage trend, city, property type, class, building size, tenant area, industry, expiry year, and rent relative to the submarket/type median. This matches the app formula.

The manual coefficient table is fitted directly with statsmodels, then compared with the app function and saved report. Odds ratio for a 10-point usage decrease = exp(-10 x usage coefficient). Ordinary app standard errors treat events as independent; a building-clustered sensitivity fit is also shown. Repeated leases, imputed rent, unnormalized rent units, outcome-label assumptions, and later-known snapshot facts limit interpretation. Convergence is checked rather than silently assumed.

In [ ]:
import statsmodels.formula.api as smf
renewal_formula="renewed ~ usage + trend + " + " + ".join(analysis._controls_for(ext_events))
print("Formula:",renewal_formula)
with warnings.catch_warnings(record=True) as renewal_fit_warnings:
    warnings.simplefilter("always")
    renewal_manual=smf.logit(renewal_formula,data=ext_events).fit(disp=0,maxiter=200)
renewal_coefficients=pd.DataFrame({"coefficient":renewal_manual.params,"std_error":renewal_manual.bse,"p_value":renewal_manual.pvalues})
ci=renewal_manual.conf_int();renewal_coefficients["coef_ci_low"]=ci[0];renewal_coefficients["coef_ci_high"]=ci[1]
display(renewal_coefficients.reset_index(names="term"))
b=float(renewal_manual.params["usage"]);se=float(renewal_manual.bse["usage"])
renewal_manual_summary={"n":int(renewal_manual.nobs),"converged":bool(renewal_manual.mle_retvals.get("converged",False)),"odds_ratio_per_10pt_drop":float(np.exp(-10*b)),"ci_low":float(np.exp(-10*(b+1.96*se))),"ci_high":float(np.exp(-10*(b-1.96*se))),"p_value":float(renewal_manual.pvalues["usage"])}
display(pd.DataFrame([renewal_manual_summary]))
print("Fit warnings:",list(dict.fromkeys(str(w.message) for w in renewal_fit_warnings)) or "none")
renewal_app_refit=analysis.logistic(ext_events)
renewal_comparison=[]
for name,mkey in [("odds_ratio_per_10pt_drop","odds_ratio_per_10pt_drop"),("or_ci_low","ci_low"),("or_ci_high","ci_high"),("p_value_usage","p_value")]:
    saved=(ext_saved_app or {}).get("results",{}).get("regression",{}).get(name)
    renewal_comparison.append({"metric":name,"manual":renewal_manual_summary[mkey],"app_refit":renewal_app_refit.get(name),"saved_app":saved})
display(pd.DataFrame(renewal_comparison))
try:
    rows=renewal_manual.model.data.row_labels
    clustered=smf.logit(renewal_formula,data=ext_events).fit(disp=0,maxiter=200,cov_type="cluster",cov_kwds={"groups":ext_events.loc[rows,"building_id"]})
    cb,cs=float(clustered.params["usage"]),float(clustered.bse["usage"])
    renewal_cluster_summary={"standard_errors":"building-clustered","odds_ratio":float(np.exp(-10*cb)),"ci_low":float(np.exp(-10*(cb+1.96*cs))),"ci_high":float(np.exp(-10*(cb-1.96*cs))),"p_value":float(clustered.pvalues["usage"])}
except Exception as error:
    renewal_cluster_summary={"error":str(error)}
display(pd.DataFrame([renewal_cluster_summary]))
ext_outputs.update(renewal_manual=renewal_manual_summary,renewal_app_refit=renewal_app_refit,renewal_clustered=renewal_cluster_summary)
# Categories with only one observed outcome can cause separation/nonconvergence.
renewal_sparse_categories=[]
for category in ["city","property_type","building_class","tenant_industry","expiry_year"]:
    cross=pd.crosstab(ext_events[category],ext_events.renewed).reindex(columns=[0,1],fill_value=0)
    for value,row in cross.iterrows():
        if row[0]==0 or row[1]==0:
            renewal_sparse_categories.append({"control":category,"category":str(value),"failures":int(row[0]),"successes":int(row[1])})
renewal_sparse_categories=pd.DataFrame(renewal_sparse_categories)
display(renewal_sparse_categories)
if not renewal_manual_summary["converged"]:
    display(Markdown("**Model warning:** the original full renewal model did not converge. Its reported odds ratio, confidence interval, and p-value are reproduction outputs, not validated estimates. One-outcome categories above are possible contributors, not a proven complete diagnosis. A converged, specified model is needed before drawing statistical conclusions."))


## 13. Test prediction on later expiry years
Train on 2019-2022 and test on 2023-2026. This is a temporal holdout, not a new-building holdout. The table counts buildings and tenants shared across periods. These tests predict the pipeline's inferred label, not verified renewal contracts.

First run the repository's original prediction function. Separately fit regularized logistic models with a training-only preprocessing pipeline that handles categories not seen in training. This second comparison is a diagnostic alternative, not an exact reproduction of the app model. No test-based tuning is performed; the probability threshold is fixed at 0.5.

AUC measures ranking (0.5 is chance ranking). Brier score and log loss assess probability error (lower is better). Accuracy can mislead if outcomes are imbalanced. The constant model uses only the training success rate. Probabilities are not ready for dashboard use merely because fitting succeeds.

**Leakage limitation:** both comparisons reuse the snapshot's usage features, including baseline construction, retrospective outcomes and eligibility. This is not a fully point-in-time backtest. Selecting the best lead time using test AUC also turns that test set into a selection set; a separate final holdout would be needed.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,brier_score_loss,log_loss,accuracy_score
prediction_original=analysis.out_of_sample(ext_events)
display(pd.DataFrame([prediction_original]))
pred_train=ext_events.loc[ext_events.expiry_year.between(2019,2022)].copy()
pred_test=ext_events.loc[ext_events.expiry_year.between(2023,2026)].copy()
prediction_split=pd.DataFrame([{"split":name,"events":len(g),"buildings":g.building_id.nunique(),"successes":int(g.renewed.sum()),"success_rate":g.renewed.mean(),"first_expiry":g.expiry_date.min(),"last_expiry":g.expiry_date.max()} for name,g in [("train",pred_train),("test",pred_test)]])
display(prediction_split)
shared_buildings=set(pred_train.building_id)&set(pred_test.building_id)
print("Buildings shared between train and test:",len(shared_buildings))
# Link model IDs to raw tenant identifiers to describe repeated-tenant exposure.
with (extended_root / "costar-hackathon-kit/data/location-occupancy.jsonl").open(encoding="utf-8") as f:
    pred_tenant_ids={str(r["id"]):r.get("tenantLocationId") for r in map(json.loads,f)}
train_tenants={pred_tenant_ids.get(str(x)) for x in pred_train.lease_id}-{None}
test_tenants={pred_tenant_ids.get(str(x)) for x in pred_test.lease_id}-{None}
print("Known tenant-location IDs shared between train and test:",len(train_tenants & test_tenants))
cat=["city","property_type","building_class"]
model_specs={"building_attributes_only":[],"building_attributes_plus_parking":["usage","trend"]}
prediction_probabilities=pd.DataFrame({"lease_id":pred_test.lease_id.to_numpy(),"building_id":pred_test.building_id.to_numpy(),"observed_inferred_success":pred_test.renewed.to_numpy()})
prediction_metrics=[]; prediction_errors=[]
if len(pred_train) and len(pred_test) and pred_train.renewed.nunique()==2 and pred_test.renewed.nunique()==2:
    prediction_probabilities["constant_training_rate"]=float(pred_train.renewed.mean())
    for name,numeric in model_specs.items():
        transformations=[("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),cat)]
        if numeric:transformations.append(("numeric",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric))
        model=Pipeline([("preprocess",ColumnTransformer(transformations)),("logistic",LogisticRegression(max_iter=2000,random_state=42))])
        try:
            with warnings.catch_warnings(record=True) as fit_warnings:
                warnings.simplefilter("always")
                model.fit(pred_train[cat+numeric],pred_train.renewed)
            prediction_probabilities[name]=model.predict_proba(pred_test[cat+numeric])[:,1]
            if fit_warnings:prediction_errors.append({"model":name,"warning":"; ".join(str(w.message) for w in fit_warnings)})
        except Exception as error:prediction_errors.append({"model":name,"error":str(error)})
    for name in prediction_probabilities.columns[3:]:
        prob=prediction_probabilities[name].to_numpy(); y=pred_test.renewed.to_numpy()
        prediction_metrics.append({"model":name,"test_events":len(y),"auc":roc_auc_score(y,prob),"brier":brier_score_loss(y,prob),"log_loss":log_loss(y,prob,labels=[0,1]),"accuracy_at_0_5":accuracy_score(y,prob>=.5)})
else:prediction_errors.append({"error":"Insufficient train/test outcome classes"})
prediction_metrics=pd.DataFrame(prediction_metrics)
display(prediction_metrics.round(4))
display(prediction_probabilities.head(15).round(4))
if prediction_errors:display(pd.DataFrame(prediction_errors))
# Calibration: compare average predicted probability with actual label frequency.
prediction_calibration=pd.DataFrame()
if "building_attributes_plus_parking" in prediction_probabilities:
    calibration=prediction_probabilities.copy()
    calibration["probability_bin"]=pd.cut(calibration.building_attributes_plus_parking,[0,.2,.4,.6,.8,1],include_lowest=True)
    prediction_calibration=calibration.groupby("probability_bin",observed=True).agg(events=("observed_inferred_success","size"),mean_prediction=("building_attributes_plus_parking","mean"),observed_rate=("observed_inferred_success","mean")).reset_index()
    display(prediction_calibration)
ext_outputs.update(prediction_original=prediction_original,prediction_metrics=prediction_metrics.to_dict("records"),prediction_split=prediction_split.to_dict("records"),shared_train_test_buildings=len(shared_buildings),shared_train_test_tenant_locations=len(train_tenants & test_tenants),prediction_errors=prediction_errors)

## 14. Run sensitivity and placebo checks
Run the repository's COVID exclusion, city/property-type subgroup regressions, lead-time sweep, and 30-shuffle renewal placebo. Small samples, singular models, nonconvergence and errors are findings to report, not evidence for the hypothesis. The repository suppresses warnings globally, so these diagnostic calls explicitly capture warnings where possible.

The renewal placebo shuffles usage and trend separately within submarkets at the event level; it is not a complete building-level randomization test. Its reported pass/fail is the author's heuristic, not a general certification. Multiple exploratory comparisons are not corrected for multiple testing. Predictions at different leads can use different eligible samples.

In [ ]:
with warnings.catch_warnings(record=True) as suite_warnings:
    warnings.simplefilter("always")
    renewal_covid=analysis.covid_check(ext_events)
    renewal_cities=analysis.subgroups(ext_events,"city")
    renewal_types=analysis.subgroups(ext_events,"property_type")
    events_by_lead={lead:events.build_lease_events(ext_b,ext_l,ext_u,lead_months=lead,window_months=18)[0] for lead in [0,12,24]}
    renewal_leads=analysis.lead_time_sweep(events_by_lead)
    renewal_placebo=analysis.placebo(ext_events,n_iter=30,seed=1)
display(pd.DataFrame([{"sample":k,**v} for k,v in renewal_covid.items()]))
display(pd.DataFrame(renewal_cities["groups"]))
display(pd.DataFrame(renewal_types["groups"]))
display(pd.DataFrame(renewal_leads["leads"]))
display(pd.DataFrame([renewal_placebo]))
suite_warning_messages=list(dict.fromkeys(str(w.message) for w in suite_warnings))
print("Validation warnings:",suite_warning_messages or "none")
ext_outputs.update(renewal_covid=renewal_covid,renewal_cities=renewal_cities,renewal_types=renewal_types,renewal_leads=renewal_leads,renewal_placebo=renewal_placebo,suite_warnings=suite_warning_messages)

## 15. Save the work and interpret the evidence
Saved files include an extended JSON report and CSV tables for occupancy, model coefficients, temporal prediction metrics, and calibration. They are produced from the current local database and can differ if that database changes.

A reproduced number is not automatically a trustworthy prediction. Outcome-label review, a point-in-time feature pipeline, and comparison against a simple baseline remain prerequisites for operational dashboard probabilities. No dashboard predictions are deployed by this notebook.

In [ ]:
from datetime import datetime,timezone
extended_output_dir=extended_root / "data/results/lease-validation"
extended_output_dir.mkdir(parents=True,exist_ok=True)
ext_outputs["generated_at_utc"]=datetime.now(timezone.utc).isoformat()
ext_outputs["data_mode"]="real"
ext_outputs["source_database"]=str(extended_root / "data/parallax_real.duckdb")
ext_outputs["event_exclusions"]=ext_exclusions
extended_json=extended_output_dir / "extended-validation-results.json"
extended_json.write_text(json.dumps(ext_outputs,indent=2,default=str),encoding="utf-8")
for name,frame in {"occupancy-audit":occupancy_audit,"occupancy-overlap-review":occupancy_overlap_review,"renewal-coefficients":renewal_coefficients.reset_index(names="term"),"prediction-metrics":prediction_metrics,"prediction-probabilities":prediction_probabilities,"prediction-calibration":prediction_calibration}.items():
    frame.to_csv(extended_output_dir / f"{name}.csv",index=False)
summary_rows=[
    {"check":"Occupancy arithmetic","result":f"{occupancy_audit.matches_app.sum()}/{len(occupancy_audit)} match","limit":"Numerator overlap and denominator quality still need review"},
    {"check":"Renewal regression","result":f"OR={renewal_manual_summary['odds_ratio_per_10pt_drop']:.3f}; 95% CI {renewal_manual_summary['ci_low']:.3f} to {renewal_manual_summary['ci_high']:.3f}","limit":"NONCONVERGED: estimates and intervals are not validated" if not renewal_manual_summary["converged"] else "Inferred labels; model assumptions still need review"},
    {"check":"Original temporal prediction","result":json.dumps(prediction_original),"limit":"Retrospective snapshot features; not a full point-in-time backtest"},
    {"check":"Occupancy placebo","result":str(occupancy_placebo.get('passed')),"limit":"30 shuffles; heuristic diagnostic"},
    {"check":"Renewal placebo","result":str(renewal_placebo.get('passed')),"limit":"Event-level shuffling within submarkets"},
]
extended_summary=pd.DataFrame(summary_rows)
display(extended_summary)
print("Saved reports and tables:",extended_output_dir)
# A brief interpretation of this run, with the important failures kept visible.
display(Markdown(
    "**Read this run cautiously:** occupancy arithmetic agreement does not resolve overlapping tenancies or missing building-area quality. "
    + ("The original full renewal regression did not converge. " if not renewal_manual_summary["converged"] else "The full renewal regression converged, but labels remain inferred. ")
    + "Compare prediction against the constant training-rate baseline as well as the building-only model. "
    + "A positive AUC lift over a weak baseline is not enough to establish useful prediction. "
    + "An event-level placebo pass does not override model or data-quality problems."
))
